## Feature: Guardrails (Self-Healing Agents)

What if the Log Analyzer produces a vague or incomplete analysis? In v1, nothing catches it — bad output flows straight to the next agent. Let's see the problem, then fix it with guardrails.

In [1]:
import os

from crewai import Agent, Crew, Process, Task
from crewai.llm import LLM
from crewai.tasks.task_output import TaskOutput
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()

llm = LLM(
    model="openai/gpt-4o",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

# A trickier log — ambiguous, short, no obvious root cause.
# The agent might produce a vague analysis here.
TRICKY_LOG_INPUT = """
[2024-01-15 09:01:22.100] INFO: Application startup sequence initiated
[2024-01-15 09:01:23.200] WARNING: Connection pool size reaching 80% capacity
[2024-01-15 09:01:24.300] WARNING: Slow query detected (2340ms) on /api/users
[2024-01-15 09:01:25.400] ERROR: Connection timeout after 30000ms
[2024-01-15 09:01:26.500] INFO: Retrying connection (attempt 2/3)
[2024-01-15 09:01:27.600] INFO: Retrying connection (attempt 3/3)
[2024-01-15 09:01:28.700] ERROR: All retry attempts exhausted
"""

In [2]:
class ErrorEntry(BaseModel):
    message: str = Field(description="The error message text")
    severity: str = Field(description="ERROR, CRITICAL, or WARNING")
    timestamp: str = Field(description="When the error occurred")


class LogAnalysisReport(BaseModel):
    primary_issue: str = Field(description="One-line description of the main issue")
    root_cause: str = Field(description="Root cause analysis based on log evidence")
    errors: list[ErrorEntry] = Field(description="All errors found in the log")
    affected_components: list[str] = Field(description="System components affected")
    timeline: list[str] = Field(description="Sequence of events leading to failure")

In [3]:
log_analyzer = Agent(
    role="DevOps Log Analyzer",
    goal="Analyze log files to identify and extract specific issues, errors, and failure patterns",
    llm=llm,
    backstory="""You are a senior DevOps engineer with 10 years of experience in 
    analyzing production logs and identifying critical issues. You excel at parsing 
    through complex log files, identifying error patterns, extracting relevant error 
    messages, and determining the root cause of failures from log data.""",
    verbose=True,
)

---

### v1 approach — no validation

We use `output_pydantic` (learned from the structured output notebook), but there is no guardrail. If the agent produces a vague report with no errors identified or a missing root cause, it passes through unchecked to the next agent.

In [4]:
v1_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="A structured log analysis report",
    output_pydantic=LogAnalysisReport,
    agent=log_analyzer,
)

v1_crew = Crew(
    agents=[log_analyzer],
    tasks=[v1_task],
    process=Process.sequential,
    verbose=False,
)

v1_result = v1_crew.kickoff(inputs={"log_data": TRICKY_LOG_INPUT})

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Task: Analyze the following log data to identify issues:                                                       │
│                                                                                                                 │
│  [2024-01-15 09:01:22.100] INFO: Application startup sequence initiated                                         │
│  [2024-01-15 09:01:23.200] WARNING: Connection pool size reaching 80% capacity                                  │
│  [2024-01-15 09:01:24.300] WARNING: Slow query detected (2340ms) on /api/users                                  │
│  [2024-01-15 09:01:25.400] ERROR: Connection timeout after 30000ms                                              │
│  [2024-01-15 09:01:26.500] INFO: Retrying connection (attempt 2/3)                                              │
│  [2024-01-15 09:01:27.600] INFO: Retrying connection (attempt 3/3)                                              │
│  [2024-01-15 09:01:28.700] ERROR: All retry attempts exhausted                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "primary_issue": "Connection failures due to timeout",                                                       │
│    "root_cause": "The primary issue seems to be the connection timeout likely caused by an insufficient         │
│  connection pool or network latency issues, exacerbated by reaching capacity limits and slow query processing   │
│  times.",                                                                                                       │
│    "errors": [                                                                                                  │
│      {                                                                                                          │
│        "message": "Connection timeout after 30000ms",                                                           │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 09:01:25.400"                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "message": "All retry attempts exhausted",                                                               │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 09:01:28.700"                                                                   │
│      }                                                                                                          │
│    ],                                                                                                           │
│    "affected_components": [                                                                                     │
│      "Connection pool",                                                                                         │
│      "Database/API"                                                                                             │
│    ],                                                                                                           │
│    "timeline": [                                                                                                │
│      "2024-01-15 09:01:22.100: Application startup sequence initiated",                                         │
│      "2024-01-15 09:01:23.200: Connection pool size reaching 80% capacity",                                     │
│      "2024-01-15 09:01:24.300: Slow query detected (2340ms) on /api/users",                                     │
│      "2024-01-15 09:01:25.400: Connection timeout after 30000ms",                                               │
│      "2024-01-15 09:01:26.500: Retrying connection (attempt 2/3)",                                              │
│      "2024-01-15 09:01:27.600: Retrying connection (attempt 3/3)",                                              │
│      "2024-01-15 09:01:28.700: All retry attempts exhausted"                                                    │
│    ]                                                   

In [5]:
# Maybe the agent found the errors, maybe it didn't.
# Either way, this output flows downstream unchecked.
report = v1_result.pydantic
print(f"Errors found: {len(report.errors)}")
print(f"Root cause: {report.root_cause}")
print(f"Affected components: {report.affected_components}")

# What if errors is empty? What if root_cause is vague?
# In v1, the Issue Investigator gets this as-is. No check. No retry.

Errors found: 2
Root cause: The primary issue seems to be the connection timeout likely caused by an insufficient connection pool or network latency issues, exacerbated by reaching capacity limits and slow query processing times.
Affected components: ['Connection pool', 'Database/API']


---

### v2 approach — add a code guardrail

A guardrail is a function that validates the output. It returns `(True, data)` to pass, or `(False, "reason")` to make the agent **retry automatically**.

Run this with `verbose=True` — you'll see the guardrail reject bad output and the agent try again.

In [8]:
from typing import Any, Tuple


def validate_log_analysis(result: TaskOutput) -> Tuple[bool, Any]:
    report = result.pydantic
    if not report or not report.errors:
        return (False, "Analysis must identify at least one error from the logs")
    if not report.root_cause:
        return (False, "Root cause analysis is required")
    return (True, report)

In [9]:
v2_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="A structured log analysis report",
    output_pydantic=LogAnalysisReport,
    guardrail=validate_log_analysis,
    agent=log_analyzer,
)

v2_crew = Crew(
    agents=[log_analyzer],
    tasks=[v2_task],
    process=Process.sequential,
    verbose=True,
)

v2_result = v2_crew.kickoff(inputs={"log_data": TRICKY_LOG_INPUT})

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2f08460e-a622-4467-931c-5b6320c83e70                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Task: Analyze the following log data to identify issues:                                                       │
│                                                                                                                 │
│  [2024-01-15 09:01:22.100] INFO: Application startup sequence initiated                                         │
│  [2024-01-15 09:01:23.200] WARNING: Connection pool size reaching 80% capacity                                  │
│  [2024-01-15 09:01:24.300] WARNING: Slow query detected (2340ms) on /api/users                                  │
│  [2024-01-15 09:01:25.400] ERROR: Connection timeout after 30000ms                                              │
│  [2024-01-15 09:01:26.500] INFO: Retrying connection (attempt 2/3)                                              │
│  [2024-01-15 09:01:27.600] INFO: Retrying connection (attempt 3/3)                                              │
│  [2024-01-15 09:01:28.700] ERROR: All retry attempts exhausted                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "primary_issue": "Connection timeout leading to failed connection retries",                                  │
│    "root_cause": "The application experienced a connection timeout which could not be resolved after retry      │
│  attempts, potentially indicating an issue with network connectivity or server-side processing.",               │
│    "errors": [                                                                                                  │
│      {                                                                                                          │
│        "message": "Connection timeout after 30000ms",                                                           │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 09:01:25.400"                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "message": "All retry attempts exhausted",                                                               │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 09:01:28.700"                                                                   │
│      }                                                                                                          │
│    ],                                                                                                           │
│    "affected_components": [                                                                                     │
│      "Connection Manager"                                                                                       │
│    ],                                                                                                           │
│    "timeline": [                                                                                                │
│      "[2024-01-15 09:01:22.100] INFO: Application startup sequence initiated",                                  │
│      "[2024-01-15 09:01:23.200] WARNING: Connection pool size reaching 80% capacity",                           │
│      "[2024-01-15 09:01:24.300] WARNING: Slow query detected (2340ms) on /api/users",                           │
│      "[2024-01-15 09:01:25.400] ERROR: Connection timeout after 30000ms",                                       │
│      "[2024-01-15 09:01:26.500] INFO: Retrying connection (attempt 2/3)",                                       │
│      "[2024-01-15 09:01:27.600] INFO: Retrying connection (attempt 3/3)",                                       │
│      "[2024-01-15 09:01:28.700] ERROR: All retry attempts exhausted"                                            │
│    ]                                                                                                            │
│  }                                                                                                              │
│                                                        

╭─────────────────────────────────────────────── 🛡️ Guardrail Check ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Evaluation Started                                                                                   │
│  Name: def validate_log_analysis(result: TaskOutput) -> T...                                                    │
│  Status: 🔄 Evaluating                                                                                          │
│  Attempt: 1                                                                                                     │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🛡️ Guardrail Success ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Passed                                                                                               │
│  Name: Validation Successful                                                                                    │
│  Status: ✅ Validated                                                                                           │
│  Attempts: 1                                                                                                    │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 55470c54-ca7a-4541-9f0f-13da7988b90c                                                                     │
│  Agent: DevOps Log Analyzer                                                                                     │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2f08460e-a622-4467-931c-5b6320c83e70                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: {                                                                                                │
│    "primary_issue": "Connection timeout leading to failed connection retries",                                  │
│    "root_cause": "The application experienced a connection timeout which could not be resolved after retry      │
│  attempts, potentially indicating an issue with network connectivity or server-side processing.",               │
│    "errors": [                                                                                                  │
│      {                                                                                                          │
│        "message": "Connection timeout after 30000ms",                                                           │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 09:01:25.400"                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "message": "All retry attempts exhausted",                                                               │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 09:01:28.700"                                                                   │
│      }                                                                                                          │
│    ],                                                                                                           │
│    "affected_components": [                                                                                     │
│      "Connection Manager"                                                                                       │
│    ],                                                                                                           │
│    "timeline": [                                                                                                │
│      "[2024-01-15 09:01:22.100] INFO: Application startup sequence initiated",                                  │
│      "[2024-01-15 09:01:23.200] WARNING: Connection pool size reaching 80% capacity",                           │
│      "[2024-01-15 09:01:24.300] WARNING: Slow query detected (2340ms) on /api/users",                           │
│      "[2024-01-15 09:01:25.400] ERROR: Connection timeout after 30000ms",                                       │
│      "[2024-01-15 09:01:26.500] INFO: Retrying connection (attempt 2/3)",                                       │
│      "[2024-01-15 09:01:27.600] INFO: Retrying connection (attempt 3/3)",                                       │
│      "[2024-01-15 09:01:28.700] ERROR: All retry attempts exhausted"                                            │
│    ]                                                                                                            │
│  }                                                    

In [10]:
# Now the output is guaranteed to have errors and a root cause
report = v2_result.pydantic
print(f"Errors found: {len(report.errors)}")
print(f"Root cause: {report.root_cause}")
for error in report.errors:
    print(f"  [{error.severity}] {error.timestamp} — {error.message}")

Errors found: 2
Root cause: The application experienced a connection timeout which could not be resolved after retry attempts, potentially indicating an issue with network connectivity or server-side processing.
  [ERROR] 2024-01-15 09:01:25.400 — Connection timeout after 30000ms
  [ERROR] 2024-01-15 09:01:28.700 — All retry attempts exhausted


---

### Bonus — no-code guardrail (plain English)

You can also write a guardrail as a plain English string. CrewAI uses an LLM to evaluate whether the output meets your criteria. No Python function needed.

In [ ]:
solution_specialist = Agent(
    role="DevOps Solution Specialist",
    goal="Provide clear, actionable solutions with step-by-step instructions",
    llm=llm,
    backstory="""You are a DevOps solutions architect who specializes in creating 
    reliable, step-by-step remediation plans for infrastructure issues.""",
    verbose=True,
)

solution_task = Task(
    description="Provide a solution for the following issue: {issue}",
    expected_output="A remediation plan with specific commands",
    guardrail="The solution must include at least 3 specific, copy-pasteable shell commands. "
    "Reject if it only contains general advice without concrete commands.",
    agent=solution_specialist,
)

solution_crew = Crew(
    agents=[solution_specialist],
    tasks=[solution_task],
    process=Process.sequential,
    verbose=True,
)

solution_result = solution_crew.kickoff(
    inputs={"issue": "Kubernetes pods failing with ImagePullBackOff due to missing registry credentials"}
)
print(f"\nFinal solution:\n{solution_result.raw}")